# 47 — Optimizers: SGD → Adam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/047-optimizers-sgd-adam.ipynb)

The same gradient, three update rules. In this notebook you minimize a one-dimensional quadratic with SGD, momentum and Adam, then plot their loss curves. The landscape is deliberately simple so the differences come from the optimizer, not the data.

In [ ]:
import numpy as np                                        # array and math helpers
target = 3.0                                              # minimum of the quadratic
results = {}                                              # loss history per rule
for name in ("sgd", "momentum", "adam"):                  # three optimizers
    w, vel, m, s = 1.0, 0.0, 0.0, 0.0                     # start values
    hist = []                                             # losses across steps
    for t in range(1, 31):                                # thirty steps
        g = w - target                                    # gradient of 0.5*(w-3)^2
        if name == "momentum":                            # momentum branch
            vel = 0.9 * vel + g                           # running gradient sum
            step = vel                                    # use the velocity
        elif name == "adam":                              # Adam branch
            m = 0.9 * m + 0.1 * g                         # first moment
            s = 0.999 * s + 0.001 * g * g                 # second moment
            mh = m / (1 - 0.9 ** t)                       # bias-corrected m
            vh = s / (1 - 0.999 ** t)                     # bias-corrected v
            step = mh / (np.sqrt(vh) + 1e-8)              # scaled direction
        else:                                             # plain SGD branch
            step = g                                      # raw gradient
        w = w - 0.1 * step                                # one update
        hist.append(0.5 * (w - target) ** 2)              # record the loss
    results[name] = hist                                  # store this run

## Read the loop

All three rules compute the same gradient `g = w - 3`. They differ only in `step`: SGD uses `g`, momentum uses a running sum of past gradients, and Adam divides the corrected mean by the corrected root-mean-square. The learning rate stays at 0.1 for all three so the comparison is fair.

`1e-8` is the epsilon guard. Without it, a zero second moment would divide by zero.

In [ ]:
import matplotlib.pyplot as plt                            # plotting library
plt.figure(figsize=(6, 3))                                 # wide figure
for name, hist in results.items():                         # one line per optimizer
    plt.plot(range(1, 31), hist, label=name)               # loss vs step
plt.yscale("log")                                          # losses span decades
plt.xlabel("step")                                        # x label
plt.ylabel("loss (log)")                                   # y label
plt.title("momentum smooths; Adam rescales")               # takeaway
plt.legend()                                               # name the lines
plt.show()                                                 # render

## What to look for

SGD approaches the minimum geometrically but can overshoot if the learning rate is raised. Momentum arrives faster and then oscillates around the target because its velocity carries it past. Adam takes a step close to the learning rate at the start — a consequence of bias correction — and then shrinks as the second moment settles.

None of this says Adam generalizes better. On a one-dimensional quadratic all three reach the same place; the differences appear on real loss surfaces with many directions.

## Your turn

Change the learning rate to 0.5 and rerun. Predict which optimizer diverges first, then check. Next, set `beta1 = 0.0` for momentum and explain why it becomes plain SGD. Finally, add a fourth branch for RMSProp — keep the second moment but use the raw gradient in the numerator — and describe its curve.